In [ ]:
############################ MEGAQUAKE FORECASTING - VECTORIZED VERSION###############################
## 6 Parametered - without Mars, but with localized zenith features and 
# a more rigorous synthetic background mining process. 
# This version also includes better error handling for missing files and more 
# informative print statements to track progress. The spatial grid scan now 
# evaluates specific latitudes for each tectonic zone, and the peak isolation 
# is applied separately for each zone to ensure we capture localized stress peaks effectively.
############################

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from sklearn.ensemble import RandomForestClassifier
from scipy.signal import find_peaks
import warnings
import math
warnings.filterwarnings('ignore')

# =====================================================================
# 🎛️ CONFIGURATION HUB
# =====================================================================
START_YEAR = 2026
FORECAST_YEARS = 5 
SCAN_INTERVAL_HOURS = 24  

MIN_PROBABILITY_ALERT = 75.0  
PEAK_ISOLATION_DAYS = 21      

HARD_NEGATIVES_COUNT = 3000   
NORMAL_BACKGROUND_COUNT = 7000 

FILE_HISTORICAL_QUAKES = '/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Inputs_Mega_Quake_Topocentric_Analysis.csv'
FILE_TECTONIC_ZONES    = '/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/subduction_zones.csv'
FILE_FORECAST_OUTPUT   = f'/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Vectorized_NoMars_Forecast_{START_YEAR}_{START_YEAR+FORECAST_YEARS}.csv'


# =====================================================================

# ==========================================
# 1. VECTORIZED PHYSICS ENGINE
# ==========================================
def calculate_julian_date(dt):
    dt = dt.astimezone(timezone.utc)
    year, month, day = dt.year, dt.month, dt.day + dt.hour/24.0 + dt.minute/1440.0 + dt.second/86400.0
    if month <= 2: year -= 1; month += 12
    A = math.floor(year / 100)
    B = 2 - A + math.floor(A / 4)
    return math.floor(365.25 * (year + 4716)) + math.floor(30.6001 * (month + 1)) + day + B - 1524.5

def get_topocentric_features_vectorized(jd, lats):
    """Calculates orbital physics for an entire array of latitudes instantly."""
    T = (jd - 2451545.0) / 36525.0
    
    # Global Alignments (Calculated ONCE per timeframe)
    sun_lon = (280.466 + 36000.77 * T + 1.915 * np.sin(np.radians(357.529 + 35999.05 * T))) % 360
    sun_dec = np.degrees(np.arcsin(np.sin(np.radians(23.439)) * np.sin(np.radians(sun_lon))))

    M_moon_deg = (134.963 + 477198.8676 * T) % 360
    moon_lon = (218.316 + 481267.88 * T + 6.289 * np.sin(np.radians(M_moon_deg))) % 360
    moon_lat = 5.128 * np.sin(np.radians(93.27 + 483202.01 * T))
    moon_dec = np.degrees(np.arcsin(np.sin(np.radians(moon_lat)) * np.cos(np.radians(23.439)) + \
              np.cos(np.radians(moon_lat)) * np.sin(np.radians(23.439)) * np.sin(np.radians(moon_lon))))
    
    def p_lon(L_m, pi, e): return (L_m + np.degrees(2 * e * np.sin(np.radians(L_m - pi)))) % 360
    sat_lon = p_lon(50.08 + 1222.1 * T, 92.06, 0.0555)
    jup_lon = p_lon(34.35 + 3034.9 * T, 14.33, 0.0485)
    
    moon_align = (1 + np.cos(np.radians(2 * (moon_lon - sun_lon)))) / 2
    sat_align = (1 + np.cos(np.radians(2 * (sat_lon - sun_lon)))) / 2
    jup_align = (1 + np.cos(np.radians(2 * (jup_lon - sun_lon)))) / 2
    lunar_dist = (1 + np.cos(np.radians(M_moon_deg))) / 2 # Perigee
    
    # Ensure lats is a numpy array for vectorization
    lats_arr = np.atleast_1d(lats)
    
    # Local Zenith Pull (Vectorized calculation across the whole grid)
    local_lunar_zenith = np.cos(np.radians(lats_arr - moon_dec))
    local_solar_zenith = np.cos(np.radians(lats_arr - sun_dec))
    
    # Broadcast global scalars to match the array size of the grid
    N = len(lats_arr)
    features_matrix = np.column_stack([
        np.full(N, moon_align),
        np.full(N, sat_align),
        np.full(N, jup_align),
        np.full(N, lunar_dist),
        local_lunar_zenith,
        local_solar_zenith
    ])
    
    # If a single scalar was passed, return a 1D array. Otherwise return the 2D matrix.
    return features_matrix[0] if np.isscalar(lats) else features_matrix

def get_peak_features_7_days_vectorized(target_dt, lats):
    """Scans 7 days and returns the peak stress for the entire grid array."""
    all_hours_matrices = []
    for h in range(0, 7*24, SCAN_INTERVAL_HOURS):
        jd = calculate_julian_date(target_dt - timedelta(hours=h))
        all_hours_matrices.append(get_topocentric_features_vectorized(jd, lats))
    
    # np.max across axis 0 flattens the 7 days into the absolute peak per coordinate
    return np.max(all_hours_matrices, axis=0)

# ==========================================
# 2. RAPID TRAINING 
# ==========================================
print("⏳ Phase 1: Training 6-Feature AI Engine...")
try:
    df_real = pd.read_csv(FILE_HISTORICAL_QUAKES)
    df_real['time'] = pd.to_datetime(df_real['time'], utc=True, format='mixed', errors='coerce')
    # Using the scalar version of our new vectorized function for single historical points
    X_real = [get_peak_features_7_days_vectorized(row['time'], row['latitude']).tolist() 
              for _, row in df_real.dropna(subset=['time', 'latitude']).iterrows()]
    y_real = [1] * len(X_real)
except FileNotFoundError:
    print(f"Error: Missing '{FILE_HISTORICAL_QUAKES}'.")
    exit()

np.random.seed(42)
start_dt = datetime(1900, 1, 1, tzinfo=timezone.utc)
total_sec = int((datetime(2025, 12, 31, tzinfo=timezone.utc) - start_dt).total_seconds())

candidate_backgrounds = []
rng = np.random.default_rng(42)
for sec in rng.choice(total_sec, size=HARD_NEGATIVES_COUNT + NORMAL_BACKGROUND_COUNT, replace=False):
    dt = start_dt + timedelta(seconds=int(sec))
    lat = np.random.uniform(-70, 70)
    feats = get_topocentric_features_vectorized(calculate_julian_date(dt), lat)
    candidate_backgrounds.append({'feats': feats, 'stress': feats[0] + feats[3] + feats[4]})

candidate_backgrounds.sort(key=lambda x: x['stress'], reverse=True)
X_syn = [item['feats'] for item in candidate_backgrounds]
y_syn = [0] * len(X_syn)

rf_model = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, n_jobs=-1, random_state=42)
rf_model.fit(np.array(X_real + X_syn), np.array(y_real + y_syn))

# ==========================================
# 3. HIGH-SPEED VECTORIZED GRID SCAN
# ==========================================
print(f"\n🚀 Phase 2: Scanning Grid ({START_YEAR}-{START_YEAR+FORECAST_YEARS}) [VECTORIZED]...")
try:
    df_grid = pd.read_csv(FILE_TECTONIC_ZONES)
    # Extract columns into fast numpy arrays
    grid_lats = df_grid['latitude'].values
    grid_lons = df_grid['longitude'].values
    grid_zones = df_grid['zone'].values
except FileNotFoundError:
    print(f"Error: '{FILE_TECTONIC_ZONES}' not found.")
    exit()

current_dt = datetime(START_YEAR, 1, 1, tzinfo=timezone.utc)
forecast_end = current_dt + timedelta(days=365 * FORECAST_YEARS)
raw_alerts = []

while current_dt <= forecast_end:
    # 1. Math engine calculates the whole grid instantly
    daily_grid_features = get_peak_features_7_days_vectorized(current_dt, grid_lats)
    
    # 2. AI predicts the whole grid instantly
    probs = rf_model.predict_proba(daily_grid_features)[:, 1] * 100
    
    # 3. Filter using fast numpy masking
    high_risk_indices = np.where(probs > MIN_PROBABILITY_ALERT)[0]
    
    for idx in high_risk_indices:
        raw_alerts.append({
            'Risk_Date': current_dt,
            'Threatened_Zone': grid_zones[idx],
            'Latitude': grid_lats[idx],
            'Longitude': grid_lons[idx], 
            'Peak_Stress_Probability': probs[idx]
        })
        
    current_dt += timedelta(hours=SCAN_INTERVAL_HOURS)
    if current_dt.month == 1 and current_dt.day == 1 and current_dt.hour == 0:
        print(f"   -> Scanned year {current_dt.year}...")

# ==========================================
# 4. EPICENTER ISOLATION
# ==========================================
if raw_alerts:
    print("\n🧹 Phase 3: Isolating Gravitational Epicenters...")
    df_raw = pd.DataFrame(raw_alerts)
    
    idx = df_raw.groupby(['Risk_Date', 'Threatened_Zone'])['Peak_Stress_Probability'].idxmax()
    df_daily_peaks = df_raw.loc[idx].sort_values('Risk_Date')
    
    clean_alerts = []
    for zone in df_daily_peaks['Threatened_Zone'].unique():
        z_data = df_daily_peaks[df_daily_peaks['Threatened_Zone'] == zone].copy()
        p_idx, _ = find_peaks(z_data['Peak_Stress_Probability'].values, distance=PEAK_ISOLATION_DAYS)
        clean_alerts.append(z_data.iloc[p_idx])
    
    df_final = pd.concat(clean_alerts).sort_values('Peak_Stress_Probability', ascending=False)
    df_final['Risk_Date'] = df_final['Risk_Date'].dt.strftime('%Y-%m-%d')
    df_final['Peak_Stress_Probability'] = df_final['Peak_Stress_Probability'].round(2)
    
    df_final = df_final[['Risk_Date', 'Threatened_Zone', 'Latitude', 'Longitude', 'Peak_Stress_Probability']]
    df_final.to_csv(FILE_FORECAST_OUTPUT, index=False)
    
    print(f"✅ Success! Accelerated run completed. Generated {len(df_final)} exact coordinate alerts.")
    print(f"💾 Map-Ready Data saved to '{FILE_FORECAST_OUTPUT}'.")
else:
    print("✅ Complete. No high-risk dates found.")

⏳ Phase 1: Training 6-Feature AI Engine...

🚀 Phase 2: Scanning Grid (2026-2031) [VECTORIZED]...
   -> Scanned year 2027...
   -> Scanned year 2028...
   -> Scanned year 2029...
   -> Scanned year 2030...
   -> Scanned year 2031...

🧹 Phase 3: Isolating Gravitational Epicenters...
✅ Success! Accelerated run completed. Generated 164 exact coordinate alerts.
💾 Map-Ready Data saved to '/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Vectorized_NoMars_Forecast_2026_2031.csv'.


In [2]:
# ==========================================
# 1. 6-VECTOR TOPOCENTRIC PHYSICS ENGINE
# ==========================================
def calculate_julian_date(dt):
    dt = dt.astimezone(timezone.utc)
    year, month, day = dt.year, dt.month, dt.day + dt.hour/24.0 + dt.minute/1440.0 + dt.second/86400.0
    if month <= 2: year -= 1; month += 12
    A = int(year / 100)
    B = 2 - A + int(A / 4)
    return int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5

def get_topocentric_features(jd, lat):
    T = (jd - 2451545.0) / 36525.0
    
    # Sun & Moon
    sun_lon = (280.466 + 36000.77 * T + 1.915 * np.sin(np.radians(357.529 + 35999.05 * T))) % 360
    sun_dec = np.degrees(np.arcsin(np.sin(np.radians(23.439)) * np.sin(np.radians(sun_lon))))

    M_moon_deg = (134.963 + 477198.8676 * T) % 360
    moon_lon = (218.316 + 481267.88 * T + 6.289 * np.sin(np.radians(M_moon_deg))) % 360
    moon_lat = 5.128 * np.sin(np.radians(93.27 + 483202.01 * T))
    moon_dec = np.degrees(np.arcsin(np.sin(np.radians(moon_lat)) * np.cos(np.radians(23.439)) + \
              np.cos(np.radians(moon_lat)) * np.sin(np.radians(23.439)) * np.sin(np.radians(moon_lon))))
    
    # Planets (Mars omitted)
    def p_lon(L_m, pi, e): return (L_m + np.degrees(2 * e * np.sin(np.radians(L_m - pi)))) % 360
    sat_lon = p_lon(50.08 + 1222.1 * T, 92.06, 0.0555)
    jup_lon = p_lon(34.35 + 3034.9 * T, 14.33, 0.0485)
    
    # Alignments
    moon_align = (1 + np.cos(np.radians(2 * (moon_lon - sun_lon)))) / 2
    sat_align = (1 + np.cos(np.radians(2 * (sat_lon - sun_lon)))) / 2
    jup_align = (1 + np.cos(np.radians(2 * (jup_lon - sun_lon)))) / 2
    
    lunar_dist = (1 + np.cos(np.radians(M_moon_deg))) / 2 # Perigee
    
    # Topocentric Zenith
    local_lunar_zenith = np.cos(np.radians(lat - moon_dec))
    local_solar_zenith = np.cos(np.radians(lat - sun_dec))
    
    # 6-Feature Array
    return [moon_align, sat_align, jup_align, lunar_dist, local_lunar_zenith, local_solar_zenith]

def get_peak_features_7_days(target_dt, lat):
    daily_features = [get_topocentric_features(calculate_julian_date(target_dt - timedelta(hours=h)), lat) 
                      for h in range(0, 7*24, 12)]
    return np.max(daily_features, axis=0).tolist()

# ==========================================
# 2. RAPID TRAINING & FEATURE IMPORTANCE
# ==========================================
print("⏳ Phase 1: Training AI Engine (WITHOUT Mars)...")
try:
    df_real = pd.read_csv('/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Inputs_Mega_Quake_Topocentric_Analysis.csv')
    df_real['time'] = pd.to_datetime(df_real['time'], utc=True, format='mixed', errors='coerce')
    X_real = [get_peak_features_7_days(row['time'], row['latitude']) for _, row in df_real.dropna(subset=['time', 'latitude']).iterrows()]
    y_real = [1] * len(X_real)
except FileNotFoundError:
    print("Error: 'Inputs_Mega_Quake_Topocentric_Analysis.csv' not found. Please ensure it is in the directory.")
    exit()

np.random.seed(42)
start_dt = datetime(1900, 1, 1, tzinfo=timezone.utc)
total_seconds = int((datetime(2025, 12, 31, tzinfo=timezone.utc) - start_dt).total_seconds())

# Quick Synthetic Background
X_syn = []
for _ in range(5000):
    dt = start_dt + timedelta(seconds=int(np.random.randint(0, total_seconds)))
    lat = np.random.uniform(-70.0, 70.0)
    X_syn.append(get_topocentric_features(calculate_julian_date(dt), lat))
y_syn = [0] * len(X_syn)

X = np.array(X_real + X_syn)
y = np.array(y_real + y_syn)

rf_model = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, random_state=42)
rf_model.fit(X, y)

# Print Feature Weights
weights = rf_model.feature_importances_
print("\n🧠 AI FEATURE IMPORTANCE (No Mars):")
print(f"   1. Lunar Syzygy:        {weights[0]*100:.1f}%")
print(f"   2. Saturn Alignment:    {weights[1]*100:.1f}%")
print(f"   3. Jupiter Alignment:   {weights[2]*100:.1f}%")
print(f"   4. Lunar Perigee:       {weights[3]*100:.1f}%")
print(f"   5. Local Lunar Zenith:  {weights[4]*100:.1f}%")
print(f"   6. Local Solar Zenith:  {weights[5]*100:.1f}%")
print("✅ AI Trained. Running Historical Diagnostics...\n")

# ==========================================
# 3. HISTORICAL DIAGNOSTICS & EXPLAINABILITY
# ==========================================
historical_test_cases = [
    {"name": "2004 Sumatra-Andaman", "time": "2004-12-26 00:58:53", "lat": 3.316},
    {"name": "2011 Tohoku (Japan)",  "time": "2011-03-11 05:46:24", "lat": 38.297},
    {"name": "2001 Bhuj (India)",    "time": "2001-01-26 03:14:40", "lat": 23.419},
    {"name": "2023 Turkey-Syria",    "time": "2023-02-06 01:17:35", "lat": 37.174}
]

print("=========================================================")
print("🚨 TOPOCENTRIC HINDCAST METRICS (NO MARS) 🚨")
print("=========================================================")

for ev in historical_test_cases:
    dt = datetime.strptime(ev['time'], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    lat = ev['lat']
    
    feats = get_peak_features_7_days(dt, lat)
    risk_probability = rf_model.predict_proba(np.array([feats]))[0][1] * 100 
    alert_level = "🔴 EXTREME" if risk_probability > 85 else "🟡 HIGH" if risk_probability > 75 else "⚪ NORMAL"
    
    print(f"🌍 Event: {ev['name']}")
    print(f"   Date: {dt.strftime('%Y-%m-%d')} | Latitude: {lat}°")
    print(f"   -> AI Output Probability: {risk_probability:.1f}% [{alert_level}]")
    print("   --- Physical Stress Metrics (0.0 to 1.0) ---")
    print(f"      • Global Lunar Syzygy:              {feats[0]:.3f}")
    print(f"      • Global Saturn Alignment:          {feats[1]:.3f}")
    print(f"      • Global Jupiter Alignment:         {feats[2]:.3f}")
    print(f"      • Lunar Distance (Perigee):         {feats[3]:.3f}")
    print(f"      • Local Lunar Zenith Lift:          {feats[4]:.3f}")
    print(f"      • Local Solar Zenith Lift:          {feats[5]:.3f}")
    print("---------------------------------------------------------")

⏳ Phase 1: Training AI Engine (WITHOUT Mars)...

🧠 AI FEATURE IMPORTANCE (No Mars):
   1. Lunar Syzygy:        44.7%
   2. Saturn Alignment:    8.0%
   3. Jupiter Alignment:   6.5%
   4. Lunar Perigee:       15.1%
   5. Local Lunar Zenith:  18.0%
   6. Local Solar Zenith:  7.6%
✅ AI Trained. Running Historical Diagnostics...

🚨 TOPOCENTRIC HINDCAST METRICS (NO MARS) 🚨
🌍 Event: 2004 Sumatra-Andaman
   Date: 2004-12-26 | Latitude: 3.316°
   -> AI Output Probability: 75.0% [⚪ NORMAL]
   --- Physical Stress Metrics (0.0 to 1.0) ---
      • Global Lunar Syzygy:              0.989
      • Global Saturn Alignment:          0.900
      • Global Jupiter Alignment:         0.019
      • Lunar Distance (Perigee):         0.626
      • Local Lunar Zenith Lift:          1.000
      • Local Solar Zenith Lift:          0.894
---------------------------------------------------------
🌍 Event: 2011 Tohoku (Japan)
   Date: 2011-03-11 | Latitude: 38.297°
   -> AI Output Probability: 88.0% [🔴 EXTREME]
   -